# Teams to Delta table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCOUNT    = "footballmanagerli"
SRC        = f"abfss://alsource@{ACCOUNT}.dfs.core.windows.net/teams/"
DEST       = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/teams/teams"
CHECKPOINT = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/_checkpoints/teams"
TABLE      = "nfl.teams"

In [0]:
team_schema = StructType([
    StructField("id",               StringType()),
    StructField("uid",              StringType()),
    StructField("slug",             StringType()),
    StructField("abbreviation",     StringType()),
    StructField("displayName",      StringType()),
    StructField("isActive",         BooleanType()),
    StructField("color",            StringType()),
    StructField("alternateColor",   StringType()),
])

file_schema = StructType([
    StructField("id",               StringType()),
    StructField("uid",              StringType()),
    StructField("slug",             StringType()),
    StructField("abbreviation",     StringType()),
    StructField("displayName",      StringType()),
    StructField("isActive",         BooleanType()),
    StructField("color",            StringType()),
    StructField("alternateColor",   StringType()),
])

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nfl")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    id                 STRING    COMMENT 'Team ID (primary key)',
    uid                STRING,
    slug               STRING,
    abbreviation       STRING,
    displayName        STRING,
    isActive           BOOLEAN,
    color              STRING,
    alternateColor     STRING,
    snapshot_at        TIMESTAMP COMMENT 'When the snapshot file was written',
    source_file        STRING,
    ingested_at        TIMESTAMP
)
USING DELTA
LOCATION '{DEST}'
""")

In [0]:
raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .schema(file_schema)
    .load(SRC)
    .select(
        "*",
        F.col("_metadata.file_path").alias("source_file"),
        F.col("_metadata.file_modification_time").alias("snapshot_at"),
    ))

teams = (raw
    .select(
        F.col("id"),
        F.col("uid"),
        F.col("slug"),
        F.col("abbreviation"),
        F.col("displayName"),
        F.col("isActive"),
        F.col("color"),
        F.col("alternateColor"),
        "snapshot_at",
        "source_file",
        F.current_timestamp().alias("ingested_at"),
    ))

In [0]:
def upsert_batch(batch_df, batch_id):
    """Upsert with automatic schema evolution: add missing columns if source has new fields."""
    
    # Step 1: Dedupe within batch (keep newest snapshot per team)
    w = Window.partitionBy("id").orderBy(F.col("snapshot_at").desc())
    latest = (batch_df
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn"))
    
    # Step 2: Auto-add missing columns (schema evolution)
    target_table = DeltaTable.forName(spark, TABLE)
    target_schema = {field.name: field.dataType for field in target_table.toDF().schema}
    batch_columns = {field.name for field in latest.schema}
    
    for col_name, col_type in target_schema.items():
        if col_name not in batch_columns:
            latest = latest.withColumn(col_name, F.lit(None).cast(col_type))
    
    # Step 3: Merge into table
    (target_table.alias("t")
        .merge(latest.alias("s"), "t.id = s.id")
        .whenMatchedUpdateAll(condition="s.snapshot_at > t.snapshot_at")
        .whenNotMatchedInsertAll()
        .execute())

(teams.writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT id) AS unique_teams FROM nfl.teams;

SELECT displayName, abbreviation, isActive FROM nfl.teams ORDER BY displayName LIMIT 20;